# DDL Constraints - Chaves Primárias e Estrangeiras

Este notebook documenta os comandos DDL para criação de **Primary Keys (PK)** e **Foreign Keys (FK)** nas tabelas da camada Gold do modelo estrela.

## Modelo de dados

```
  dim_curso (pk_curso)       dim_egresso (pk_egresso)       dim_emprego (pk_emprego)
       │                          │                              │
       └──────────┬───────────────┘                              │
                  ▼                                              │
              fato_egressos ───────────────────────────────────┘
                  (3 FKs)
```

## Regras
- As PKs devem ser criadas **antes** das FKs que as referenciam
- Colunas de PK devem ser `NOT NULL`
- As constraints são **informativas** (não há enforced pelo Databricks)
- O diagrama ERD é visível no Catalog Explorer após a criação das FKs

In [0]:
# ============================================================
# 1. PRIMARY KEYS - Tabelas de Dimensão
# ============================================================
# Idempotente: so adiciona a PK se ainda nao existir

pk_definitions = [
    ("mvp_eng_dados.gold.dim_curso",   "pk_curso",   "id_curso"),
    ("mvp_eng_dados.gold.dim_egresso", "pk_egresso", "id_egresso"),
    ("mvp_eng_dados.gold.dim_emprego", "pk_emprego", "id_emprego"),
]

for table, constraint, column in pk_definitions:
    spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} SET NOT NULL")
    try:
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {constraint} PRIMARY KEY ({column})")
        print(f"PK {constraint} criada em {table}({column})")
    except Exception as e:
        if "already exists" in str(e):
            print(f"PK {constraint} ja existe em {table}({column}) - ignorando")
        else:
            raise

In [0]:
# ============================================================
# 2. FOREIGN KEYS - Tabela de Fatos (fato_egressos)
# ============================================================
# Idempotente: so adiciona a FK se ainda nao existir

fk_definitions = [
    ("mvp_eng_dados.gold.fato_egressos", "fk_egressos_curso",   "id_curso",   "mvp_eng_dados.gold.dim_curso"),
    ("mvp_eng_dados.gold.fato_egressos", "fk_egressos_egresso", "id_egresso", "mvp_eng_dados.gold.dim_egresso"),
    ("mvp_eng_dados.gold.fato_egressos", "fk_egressos_emprego", "id_emprego", "mvp_eng_dados.gold.dim_emprego"),
]

for table, constraint, fk_col, ref_table in fk_definitions:
    try:
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {constraint} FOREIGN KEY ({fk_col}) REFERENCES {ref_table}")
        print(f"FK {constraint} criada: {table}.{fk_col} -> {ref_table}")
    except Exception as e:
        if "already exists" in str(e):
            print(f"FK {constraint} ja existe em {table} - ignorando")
        else:
            raise

In [0]:
%sql
-- ============================================================
-- 3. VERIFICAÇÃO - Listar constraints criadas
-- ============================================================

-- Verificar constraints da tabela de fatos
DESCRIBE TABLE EXTENDED mvp_eng_dados.gold.fato_egressos;

-- Verificar constraints das dimensões
DESCRIBE TABLE EXTENDED mvp_eng_dados.gold.dim_curso;
DESCRIBE TABLE EXTENDED mvp_eng_dados.gold.dim_egresso;
DESCRIBE TABLE EXTENDED mvp_eng_dados.gold.dim_emprego;